# 08. Comparación experimental de arquitecturas para Retinopatía Diabética

Este notebook realiza la comparación final entre las arquitecturas
candidatas ResNet50 y EfficientNetB3+SE utilizando exclusivamente el
conjunto Validation previamente congelado.

Los candidatos utilizados corresponden al mejor checkpoint seleccionado
dentro de cada arquitectura:

- ResNet50: Fine-Tuning.
- EfficientNetB3+SE: Fine-Tuning.

El criterio de selección fue definido antes de realizar la comparación:

1. Quadratic Weighted Kappa (QWK) como métrica principal.
2. Macro F1 como métrica secundaria.
3. Accuracy, Macro Precision y Macro Recall como métricas descriptivas.
4. Complejidad computacional como criterio complementario.

El conjunto Test permanece aislado y solo será utilizado después de
seleccionar definitivamente una arquitectura.

## 1. Importación de dependencias y rutas de resultados

Se importan las bibliotecas para cálculo de métricas de concordancia y desempeño, y se configuran las rutas absolutas hacia los directorios de resultados de ambos modelos candidatos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score
)

PROJECT_ROOT = Path(
    "/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan"
)
RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
    / "RD"
)
RESNET_RESULTS_DIR = (
    RESULTS_ROOT
    / "resnet50"
)
EFFICIENTNET_RESULTS_DIR = (
    RESULTS_ROOT
    / "efficientnetb3_se"
)
COMPARISON_RESULTS_DIR = (
    RESULTS_ROOT
    / "model_comparison"
)
COMPARISON_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESNET_PREDICTIONS_PATH = (
    RESNET_RESULTS_DIR
    / "resnet50_finetune_validation_predictions.csv"
)
EFFICIENTNET_PREDICTIONS_PATH = (
    EFFICIENTNET_RESULTS_DIR
    / "efficientnetb3_se_finetune_validation_predictions.csv"
)

print(
    "ResNet:",
    RESNET_PREDICTIONS_PATH.exists()
)
print(
    "EfficientNet:",
    EFFICIENTNET_PREDICTIONS_PATH.exists()
)

ResNet: True
EfficientNet: True


## 2. Verificación de integridad de las predicciones de Validation

Se cargan los archivos de predicciones generados por ambos modelos y se comprueba que contengan exactamente 523 registros sin duplicados.

In [2]:
resnet_df = pd.read_csv(
    RESNET_PREDICTIONS_PATH
)
efficientnet_df = pd.read_csv(
    EFFICIENTNET_PREDICTIONS_PATH
)

print(
    "Filas ResNet:",
    len(resnet_df)
)
print(
    "Filas EfficientNet:",
    len(efficientnet_df)
)
print(
    "IDs duplicados ResNet:",
    resnet_df["id_code"].duplicated().sum()
)
print(
    "IDs duplicados EfficientNet:",
    efficientnet_df["id_code"].duplicated().sum()
)

Filas ResNet: 523
Filas EfficientNet: 523
IDs duplicados ResNet: 0
IDs duplicados EfficientNet: 0


### 2.1. Comprobación de equivalencia estricta de identificadores

Se comprueba que ambos conjuntos contengan exactamente el mismo conjunto de identificadores `id_code`, garantizando una evaluación pareada estricta.

In [3]:
print(
    "Mismos IDs:",
    set(
        resnet_df["id_code"]
    )
    ==
    set(
        efficientnet_df["id_code"]
    )
)

Mismos IDs: True


## 3. Unificación de predicciones sobre Validation

Las predicciones generadas por ambos modelos se integran utilizando
`id_code` como identificador.

Antes de calcular las métricas comparativas se verifica que las etiquetas
reales asociadas a cada imagen coincidan entre ambos archivos.

In [4]:
comparison_df = resnet_df.merge(
    efficientnet_df,
    on="id_code",
    how="inner",
    suffixes=(
        "_resnet",
        "_efficientnet"
    )
)

print(
    "Filas después de unir:",
    len(comparison_df)
)

labels_match = np.array_equal(
    comparison_df[
        "diagnosis_resnet"
    ].to_numpy(),
    comparison_df[
        "diagnosis_efficientnet"
    ].to_numpy()
)

print(
    "Etiquetas coinciden:",
    labels_match
)

if not labels_match:
    raise ValueError(
        "Las etiquetas reales no coinciden."
    )

Filas después de unir: 523
Etiquetas coinciden: True


### 3.1. Estructuración y persistencia de la tabla unificada

Se consolida un DataFrame limpio con los identificadores, etiquetas reales y predicciones discretas de ambos modelos para la comparación pareada.

In [5]:
comparison_scores_df = pd.DataFrame(
    {
        "id_code":
            comparison_df["id_code"],
        "label":
            comparison_df[
                "diagnosis_resnet"
            ],
        "resnet50_prediction":
            comparison_df[
                "predicted_class_resnet"
            ],
        "efficientnetb3_se_prediction":
            comparison_df[
                "predicted_class_efficientnet"
            ]
    }
)

display(
    comparison_scores_df.head()
)

COMBINED_PREDICTIONS_PATH = (
    COMPARISON_RESULTS_DIR
    / "rd_validation_model_predictions.csv"
)
comparison_scores_df.to_csv(
    COMBINED_PREDICTIONS_PATH,
    index=False
)

print(
    "Comparación guardada en:",
    COMBINED_PREDICTIONS_PATH
)

,id_code,label,resnet50_prediction,efficientnetb3_se_prediction
0,0024cdab0c1e,1,1,1
1,0083ee8054ee,4,4,2
2,00a8624548a9,2,2,2
3,00b74780d31d,2,2,2
4,01b3aed3ed4c,1,1,1


Comparación guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/RD/model_comparison/rd_validation_model_predictions.csv


## 4. Recálculo unificado de métricas sobre Validation

Se recalculan las métricas comparativas directamente sobre el DataFrame consolidado para asegurar total consistencia y orden estricto de los vectores evaluados.

In [6]:
y_val = comparison_scores_df[
    "label"
].to_numpy()
pred_resnet = comparison_scores_df[
    "resnet50_prediction"
].to_numpy()
pred_efficientnet = comparison_scores_df[
    "efficientnetb3_se_prediction"
].to_numpy()


def calculate_metrics(
    y_true,
    y_pred
):
    return {
        "QWK":
            cohen_kappa_score(
                y_true,
                y_pred,
                weights="quadratic"
            ),
        "Macro F1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
        "Accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),
        "Macro Precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),
        "Macro Recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
    }


resnet_metrics = calculate_metrics(
    y_val,
    pred_resnet
)
efficientnet_metrics = calculate_metrics(
    y_val,
    pred_efficientnet
)

### 4.1. Construcción de la tabla comparativa de desempeño

Se enfrentan las métricas calculadas y se determina la diferencia aritmética (`EfficientNetB3+SE - ResNet50`) para cuantificar la ventaja entre arquitecturas.

In [7]:
exact_comparison_df = pd.DataFrame(
    {
        "ResNet50": resnet_metrics,
        "EfficientNetB3+SE":
            efficientnet_metrics
    }
)
exact_comparison_df[
    "Diferencia EfficientNet - ResNet"
] = (
    exact_comparison_df[
        "EfficientNetB3+SE"
    ]
    - exact_comparison_df[
        "ResNet50"
    ]
)

display(
    exact_comparison_df.round(6)
)

exact_comparison_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "rd_validation_exact_comparison.csv"
)

,ResNet50,EfficientNetB3+SE,Diferencia EfficientNet - ResNet
QWK,0.867617,0.877946,0.010330
Macro F1,0.623143,0.618500,-0.004643
Accuracy,0.793499,0.780115,-0.013384
Macro Precision,0.616926,0.643499,0.026573
Macro Recall,0.647538,0.648837,0.001299


## 5. Análisis de incertidumbre mediante bootstrap pareado estratificado

Debido a que ambos modelos fueron evaluados sobre exactamente las mismas
523 imágenes de Validation, se utiliza un bootstrap pareado para estimar
la incertidumbre de las diferencias observadas.

El remuestreo se realiza de manera estratificada según la clase verdadera,
preservando la composición original de las cinco categorías en cada
repetición.

Se utilizan 5000 repeticiones con semilla fija.

El análisis se concentra principalmente en:

- diferencia de QWK, métrica principal;
- diferencia de Macro F1, métrica secundaria.

In [8]:
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

class_indices = {
    c: np.where(y_val == c)[0]
    for c in np.unique(y_val)
}

bootstrap_rows = []

for _ in range(N_BOOTSTRAP):
    boot_indices = np.concatenate(
        [
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices.values()
        ]
    )

    y_boot = y_val[boot_indices]
    resnet_boot = pred_resnet[boot_indices]
    efficient_boot = pred_efficientnet[boot_indices]

    qwk_resnet = cohen_kappa_score(
        y_boot,
        resnet_boot,
        weights="quadratic"
    )
    qwk_efficient = cohen_kappa_score(
        y_boot,
        efficient_boot,
        weights="quadratic"
    )
    f1_resnet = f1_score(
        y_boot,
        resnet_boot,
        average="macro",
        zero_division=0
    )
    f1_efficient = f1_score(
        y_boot,
        efficient_boot,
        average="macro",
        zero_division=0
    )

    bootstrap_rows.append(
        {
            "qwk_resnet":
                qwk_resnet,
            "qwk_efficientnet":
                qwk_efficient,
            "qwk_difference":
                qwk_efficient
                - qwk_resnet,
            "macro_f1_resnet":
                f1_resnet,
            "macro_f1_efficientnet":
                f1_efficient,
            "macro_f1_difference":
                f1_efficient
                - f1_resnet
        }
    )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

print(
    "Repeticiones realizadas:",
    len(bootstrap_df)
)

Repeticiones realizadas: 5000


### 5.1. Estimación de intervalos de confianza del 95 %

Se computan los percentiles 2.5 y 97.5 para QWK, Macro F1 y las diferencias pareadas entre EfficientNetB3+SE y ResNet50.

In [9]:
def ci_95(values):
    return np.percentile(
        values,
        [
            2.5,
            97.5
        ]
    )


qwk_resnet_ci = ci_95(
    bootstrap_df[
        "qwk_resnet"
    ]
)
qwk_efficient_ci = ci_95(
    bootstrap_df[
        "qwk_efficientnet"
    ]
)
qwk_difference_ci = ci_95(
    bootstrap_df[
        "qwk_difference"
    ]
)
f1_resnet_ci = ci_95(
    bootstrap_df[
        "macro_f1_resnet"
    ]
)
f1_efficient_ci = ci_95(
    bootstrap_df[
        "macro_f1_efficientnet"
    ]
)
f1_difference_ci = ci_95(
    bootstrap_df[
        "macro_f1_difference"
    ]
)

### 5.2. Reporte de intervalos de confianza e incertidumbre

Se despliegan las estimaciones puntuales con sus intervalos de confianza del 95% para evaluar la significancia estadística de la ventaja observada.

In [10]:
print(
    f"QWK ResNet50: "
    f"{resnet_metrics['QWK']:.6f} "
    f"[IC95%: "
    f"{qwk_resnet_ci[0]:.6f}, "
    f"{qwk_resnet_ci[1]:.6f}]"
)
print(
    f"QWK EfficientNetB3+SE: "
    f"{efficientnet_metrics['QWK']:.6f} "
    f"[IC95%: "
    f"{qwk_efficient_ci[0]:.6f}, "
    f"{qwk_efficient_ci[1]:.6f}]"
)
print(
    f"Diferencia QWK "
    f"(EfficientNet - ResNet): "
    f"{efficientnet_metrics['QWK'] - resnet_metrics['QWK']:.6f} "
    f"[IC95%: "
    f"{qwk_difference_ci[0]:.6f}, "
    f"{qwk_difference_ci[1]:.6f}]"
)
print()
print(
    f"Macro F1 ResNet50: "
    f"{resnet_metrics['Macro F1']:.6f} "
    f"[IC95%: "
    f"{f1_resnet_ci[0]:.6f}, "
    f"{f1_resnet_ci[1]:.6f}]"
)
print(
    f"Macro F1 EfficientNetB3+SE: "
    f"{efficientnet_metrics['Macro F1']:.6f} "
    f"[IC95%: "
    f"{f1_efficient_ci[0]:.6f}, "
    f"{f1_efficient_ci[1]:.6f}]"
)
print(
    f"Diferencia Macro F1 "
    f"(EfficientNet - ResNet): "
    f"{efficientnet_metrics['Macro F1'] - resnet_metrics['Macro F1']:.6f} "
    f"[IC95%: "
    f"{f1_difference_ci[0]:.6f}, "
    f"{f1_difference_ci[1]:.6f}]"
)

bootstrap_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "rd_bootstrap_validation_differences.csv",
    index=False
)

QWK ResNet50: 0.867617 [IC95%: 0.836131, 0.896260]
QWK EfficientNetB3+SE: 0.877946 [IC95%: 0.847768, 0.906084]
Diferencia QWK (EfficientNet - ResNet): 0.010330 [IC95%: -0.015200, 0.036515]

Macro F1 ResNet50: 0.623143 [IC95%: 0.571106, 0.676442]
Macro F1 EfficientNetB3+SE: 0.618500 [IC95%: 0.566791, 0.671238]
Diferencia Macro F1 (EfficientNet - ResNet): -0.004643 [IC95%: -0.061075, 0.050741]


## 6. Selección de la arquitectura

La comparación directa sobre las 523 imágenes de Validation mostró un
rendimiento cercano entre ResNet50 y EfficientNetB3+SE.

ResNet50 obtuvo un QWK de 0.867617 y un Macro F1 de 0.623143, mientras
que EfficientNetB3+SE obtuvo un QWK de 0.877946 y un Macro F1 de
0.618500.

La diferencia puntual de QWK a favor de EfficientNetB3+SE fue de
0.010330. Sin embargo, el intervalo de confianza del 95 % de dicha
diferencia, estimado mediante bootstrap pareado estratificado, fue
[-0.015200, 0.036515], incluyendo el valor cero.

De manera similar, la diferencia de Macro F1 fue de -0.004643, con un
intervalo de confianza del 95 % de [-0.061075, 0.050741].

Por tanto, los resultados de Validation no evidencian una diferencia
clara entre ambas arquitecturas dentro de la incertidumbre del conjunto
evaluado.

No obstante, el criterio experimental fue definido previamente
estableciendo QWK como métrica principal. EfficientNetB3+SE presentó la
mayor estimación puntual de QWK y, adicionalmente, una menor cantidad
de parámetros que ResNet50.

En consecuencia, se selecciona EfficientNetB3+SE con Fine-Tuning como
arquitectura final del experimento de Retinopatía Diabética.

Esta selección se limita a las condiciones experimentales y al conjunto
Validation utilizado y no implica superioridad general de la
arquitectura.

## 7. Persistencia del análisis bootstrap y resumen comparativo

Se guardan las réplicas bootstrap y se genera un DataFrame con las estimaciones puntuales de QWK y Macro F1 e intervalos de confianza del 95% para las diferencias entre modelos.

In [11]:
bootstrap_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "rd_validation_paired_bootstrap.csv",
    index=False
)

bootstrap_summary_df = pd.DataFrame(
    [
        {
            "metric": "QWK",
            "resnet50": resnet_metrics["QWK"],
            "efficientnetb3_se":
                efficientnet_metrics["QWK"],
            "difference_efficient_minus_resnet":
                efficientnet_metrics["QWK"]
                - resnet_metrics["QWK"],
            "difference_ci95_lower":
                qwk_difference_ci[0],
            "difference_ci95_upper":
                qwk_difference_ci[1]
        },
        {
            "metric": "Macro F1",
            "resnet50":
                resnet_metrics["Macro F1"],
            "efficientnetb3_se":
                efficientnet_metrics["Macro F1"],
            "difference_efficient_minus_resnet":
                efficientnet_metrics["Macro F1"]
                - resnet_metrics["Macro F1"],
            "difference_ci95_lower":
                f1_difference_ci[0],
            "difference_ci95_upper":
                f1_difference_ci[1]
        }
    ]
)

display(
    bootstrap_summary_df.round(6)
)

bootstrap_summary_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "rd_validation_bootstrap_summary.csv",
    index=False
)

,metric,resnet50,efficientnetb3_se,difference_efficient_minus_resnet,difference_ci95_lower,difference_ci95_upper
0,QWK,0.867617,0.877946,0.010330,-0.015200,0.036515
1,Macro F1,0.623143,0.618500,-0.004643,-0.061075,0.050741


## 8. Congelamiento formal del modelo seleccionado

Se registra y persiste la configuración final del modelo ganador (EfficientNetB3+SE con Fine-Tuning), consolidando sus métricas en Validation y la confirmación explícita de que Test se mantuvo intacto durante la decisión.

In [12]:
import json

SELECTED_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "RD"
    / "efficientnetb3_se"
    / "efficientnetb3_se_finetune_best.keras"
)

FINAL_CONFIG_PATH = (
    COMPARISON_RESULTS_DIR
    / "rd_selected_model_config.json"
)

final_model_config = {
    "selected_architecture":
        "EfficientNetB3+SE",
    "checkpoint_phase":
        "Fine-Tuning",
    "checkpoint_path":
        str(SELECTED_MODEL_PATH),
    "selection_criteria":
        "Highest Validation QWK and lower parameter count",
    "validation_metrics": {
        "QWK":
            float(
                efficientnet_metrics["QWK"]
            ),
        "Macro F1":
            float(
                efficientnet_metrics["Macro F1"]
            ),
        "Accuracy":
            float(
                efficientnet_metrics["Accuracy"]
            ),
        "Macro Precision":
            float(
                efficientnet_metrics[
                    "Macro Precision"
                ]
            ),
        "Macro Recall":
            float(
                efficientnet_metrics[
                    "Macro Recall"
                ]
            )
    },
    "qwk_difference_vs_resnet":
        float(
            efficientnet_metrics["QWK"]
            - resnet_metrics["QWK"]
        ),
    "qwk_difference_ci95": [
        float(qwk_difference_ci[0]),
        float(qwk_difference_ci[1])
    ],
    "bootstrap_repetitions":
        N_BOOTSTRAP,
    "test_used_for_selection":
        False
}

with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_model_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Modelo seleccionado:",
    final_model_config[
        "selected_architecture"
    ]
)
print(
    "Checkpoint:",
    SELECTED_MODEL_PATH
)
print(
    "Configuración guardada en:",
    FINAL_CONFIG_PATH
)

Modelo seleccionado: EfficientNetB3+SE
Checkpoint: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/models/RD/efficientnetb3_se/efficientnetb3_se_finetune_best.keras
Configuración guardada en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/RD/model_comparison/rd_selected_model_config.json


## 9. Evaluación final sobre Test

Una vez seleccionada y congelada la arquitectura utilizando
exclusivamente Validation, se realiza la evaluación final sobre el
conjunto Test.

Test contiene 524 imágenes y no fue utilizado durante el entrenamiento,
Fine-Tuning, selección de checkpoints ni comparación de arquitecturas.

Esta evaluación constituye la estimación final del rendimiento del
modelo seleccionado dentro del protocolo experimental definido.

### 9.1. Carga y verificación de la partición Test

Se recuperan las 524 imágenes de Test desde el manifiesto de datos depurados, comprobando la composición por clases de la partición aislada.

In [13]:
METADATA_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "RD"
    / "aptos2019"
    / "metadata"
)
APTOS_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "RD"
    / "aptos2019"
)
MANIFEST_PATH = (
    METADATA_DIR
    / "aptos2019_processed_manifest.csv"
)

manifest_df = pd.read_csv(
    MANIFEST_PATH
)
manifest_df[
    "absolute_processed_path"
] = manifest_df[
    "processed_path"
].apply(
    lambda x:
        str(APTOS_ROOT / x)
)

test_df = (
    manifest_df[
        manifest_df[
            "experimental_split"
        ] == "test"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Imágenes Test:",
    len(test_df)
)
print(
    "\nDistribución:"
)
print(
    test_df[
        "diagnosis"
    ]
    .value_counts()
    .sort_index()
)

Imágenes Test: 524

Distribución:
diagnosis
0    270
1     51
2    138
3     25
4     40
Name: count, dtype: int64


### 9.2. Construcción del dataset de evaluación para Test

Se construye el flujo de entrada `tf.data.Dataset` sin alteración ni aumentación de datos para la evaluación objetiva de Test.

In [14]:
import tensorflow as tf

IMAGE_SIZE = 384
BATCH_SIZE = 8

def load_image_and_label(
    image_path,
    label
):
    image = tf.io.read_file(
        image_path
    )
    image = tf.image.decode_png(
        image,
        channels=3
    )
    image = tf.cast(
        image,
        tf.float32
    )
    image.set_shape(
        [
            IMAGE_SIZE,
            IMAGE_SIZE,
            3
        ]
    )
    return (
        image,
        tf.cast(
            label,
            tf.int32
        )
    )

def build_dataset(
    dataframe,
    training=False
):
    paths = dataframe[
        "absolute_processed_path"
    ].to_numpy()
    labels = dataframe[
        "diagnosis"
    ].to_numpy(
        dtype=np.int32
    )

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                paths,
                labels
            )
        )
    )

    dataset = dataset.map(
        load_image_and_label,
        num_parallel_calls=2
    )
    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=False
    )
    dataset = dataset.prefetch(
        1
    )

    return dataset

test_ds = build_dataset(
    test_df,
    training=False
)

print(
    "Dataset Test construido con",
    len(test_df),
    "imágenes."
)

I0000 00:00:1790759532.027118  275542 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790759532.329024  275542 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790759534.355970  275542 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1790759538.446711  275542 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel bi

Dataset Test construido con 524 imágenes.


### 9.3. Carga del checkpoint óptimo del modelo seleccionado

Se carga en memoria el modelo ganador congelado (`efficientnetb3_se_finetune_best.keras`) sin recompilar.

In [15]:
final_model = tf.keras.models.load_model(
    SELECTED_MODEL_PATH,
    compile=False
)
print(
    "Modelo final cargado:",
    SELECTED_MODEL_PATH
)

Modelo final cargado: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/models/RD/efficientnetb3_se/efficientnetb3_se_finetune_best.keras


## 10. Predicción final sobre Test

Se realiza la única evaluación del modelo seleccionado sobre las 524
imágenes del conjunto Test.

La arquitectura, los pesos y todas las decisiones experimentales fueron
congelados previamente utilizando exclusivamente Train y Validation.

Los resultados obtenidos en Test se utilizan únicamente para estimar el
rendimiento final del modelo y no para realizar nuevos ajustes.

### 10.1. Generación de predicciones continuas sobre Test

Se obtienen las distribuciones de probabilidad de salida para las 524 imágenes de Test.

In [16]:
test_probabilities = final_model.predict(
    test_ds,
    verbose=1
)
print(
    "Shape probabilidades:",
    test_probabilities.shape
)
print(
    "Suma primera predicción:",
    test_probabilities[0].sum()
)

I0000 00:00:1790759794.115830  277172 cuda_dnn.cc:461] Loaded cuDNN version 92600


66/66 ━━━━━━━━━━━━━━━━━━━━ 17s 74ms/step
Shape probabilidades: (524, 5)
Suma primera predicción: 1.0


### 10.2. Extracción de etiquetas reales y clases predichas en Test

Se extraen las etiquetas ground truth reales del dataset y se calcula la clase de máxima probabilidad (`argmax`) por cada muestra.

In [17]:
y_test = np.concatenate(
    [
        labels.numpy()
        for _, labels in test_ds
    ]
)
y_pred_test = np.argmax(
    test_probabilities,
    axis=1
)

print(
    "Labels reales:",
    y_test.shape
)
print(
    "Predicciones:",
    y_pred_test.shape
)
print(
    "Clases predichas:",
    sorted(
        np.unique(
            y_pred_test
        ).tolist()
    )
)

Labels reales: (524,)
Predicciones: (524,)
Clases predichas: [0, 1, 2, 3, 4]


## 11. Métricas finales

El rendimiento final se cuantifica utilizando las mismas métricas
predefinidas durante la comparación experimental:

- Quadratic Weighted Kappa (QWK);
- Macro F1;
- Accuracy;
- Macro Precision;
- Macro Recall;
- matriz de confusión;
- métricas individuales por clase.

In [19]:
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

print("Métricas de sklearn importadas correctamente.")

Métricas de sklearn importadas correctamente.


In [20]:
test_qwk = cohen_kappa_score(
    y_test,
    y_pred_test,
    weights="quadratic"
)
test_macro_f1 = f1_score(
    y_test,
    y_pred_test,
    average="macro",
    zero_division=0
)
test_accuracy = accuracy_score(
    y_test,
    y_pred_test
)
test_macro_precision = precision_score(
    y_test,
    y_pred_test,
    average="macro",
    zero_division=0
)
test_macro_recall = recall_score(
    y_test,
    y_pred_test,
    average="macro",
    zero_division=0
)
test_cm = confusion_matrix(
    y_test,
    y_pred_test,
    labels=[0, 1, 2, 3, 4]
)

print(
    f"QWK:             {test_qwk:.6f}"
)
print(
    f"Macro F1:        {test_macro_f1:.6f}"
)
print(
    f"Accuracy:        {test_accuracy:.6f}"
)
print(
    f"Macro Precision: {test_macro_precision:.6f}"
)
print(
    f"Macro Recall:    {test_macro_recall:.6f}"
)
print(
    "\nMatriz de confusión:"
)
print(test_cm)

QWK:             0.860161
Macro F1:        0.626650
Accuracy:        0.776718
Macro Precision: 0.668083
Macro Recall:    0.660466

Matriz de confusión:
[[260   9   1   0   0]
 [  2  43   6   0   0]
 [  1  47  74  14   2]
 [  0   3   6  14   2]
 [  0   7   8   9  16]]


## 12. Rendimiento detallado por clase diagnóstica en Test

Se evalúa la precisión, sensibilidad (recall) y F1-score discriminados para cada una de las cinco categorías diagnósticas de retinopatía diabética, prestando especial atención a las clases minoritarias.

In [21]:
from sklearn.metrics import classification_report

test_report = classification_report(
    y_test,
    y_pred_test,
    labels=[0, 1, 2, 3, 4],
    target_names=[
        "Sin RD",
        "Leve",
        "Moderada",
        "Severa",
        "Proliferativa"
    ],
    output_dict=True,
    zero_division=0
)
test_report_df = (
    pd.DataFrame(
        test_report
    )
    .transpose()
)
display(
    test_report_df.round(4)
)

,precision,recall,f1-score,support
Sin RD,0.9886,0.9630,0.9756,270.0000
Leve,0.3945,0.8431,0.5375,51.0000
Moderada,0.7789,0.5362,0.6352,138.0000
Severa,0.3784,0.5600,0.4516,25.0000
Proliferativa,0.8000,0.4000,0.5333,40.0000
accuracy,0.7767,0.7767,0.7767,0.7767
macro avg,0.6681,0.6605,0.6266,524.0000
weighted avg,0.8320,0.7767,0.7846,524.0000


## 13. Matriz de confusión tabular sobre Test

Se estructura la matriz de confusión multiclase sobre las 524 muestras de Test para visualizar los aciertos y las confusiones ordinales entre grados de severidad adyacentes.

In [22]:
test_cm_df = pd.DataFrame(
    test_cm,
    index=[
        "True_0",
        "True_1",
        "True_2",
        "True_3",
        "True_4"
    ],
    columns=[
        "Pred_0",
        "Pred_1",
        "Pred_2",
        "Pred_3",
        "Pred_4"
    ]
)
display(
    test_cm_df
)

,Pred_0,Pred_1,Pred_2,Pred_3,Pred_4
True_0,260,9,1,0,0
True_1,2,43,6,0,0
True_2,1,47,74,14,2
True_3,0,3,6,14,2
True_4,0,7,8,9,16


## 14. Persistencia de predicciones y métricas de Test

Se almacenan los vectores de probabilidad Softmax continuos y las clases predichas de Test en formato CSV, junto con las métricas escalares, el reporte por clase y la matriz de confusión.

In [23]:
test_results_df = (
    test_df[
        [
            "id_code",
            "diagnosis"
        ]
    ]
    .copy()
)
test_results_df[
    "predicted_class"
] = y_pred_test

for class_id in range(5):
    test_results_df[
        f"score_class_{class_id}"
    ] = test_probabilities[
        :,
        class_id
    ]

TEST_SCORES_PATH = (
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_predictions.csv"
)
test_results_df.to_csv(
    TEST_SCORES_PATH,
    index=False
)
print(
    "Predicciones Test guardadas en:",
    TEST_SCORES_PATH
)

test_metrics_df = pd.DataFrame(
    {
        "metric": [
            "QWK",
            "Macro F1",
            "Accuracy",
            "Macro Precision",
            "Macro Recall"
        ],
        "value": [
            test_qwk,
            test_macro_f1,
            test_accuracy,
            test_macro_precision,
            test_macro_recall
        ]
    }
)
display(
    test_metrics_df.round(6)
)
test_metrics_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_metrics.csv",
    index=False
)
test_report_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_classification_report.csv"
)
test_cm_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_confusion_matrix.csv"
)

Predicciones Test guardadas en: /mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/RD/model_comparison/efficientnetb3_se_test_predictions.csv


,metric,value
0,QWK,0.860161
1,Macro F1,0.626650
2,Accuracy,0.776718
3,Macro Precision,0.668083
4,Macro Recall,0.660466


## 15. Intervalos de confianza mediante bootstrap estratificado

Para cuantificar la incertidumbre asociada a las métricas finales de Test
se utiliza bootstrap estratificado por clase.

Cada repetición conserva la composición original de las cinco categorías
diagnósticas.

Se realizan 5000 repeticiones con semilla fija.

In [24]:
N_TEST_BOOTSTRAP = 5000
BOOTSTRAP_TEST_SEED = 42
rng_test = np.random.default_rng(
    BOOTSTRAP_TEST_SEED
)

class_indices_test = {
    c: np.where(y_test == c)[0]
    for c in np.unique(y_test)
}

bootstrap_test_rows = []

for _ in range(N_TEST_BOOTSTRAP):
    sampled_indices = []
    for c, indices in class_indices_test.items():
        sampled_class = rng_test.choice(
            indices,
            size=len(indices),
            replace=True
        )
        sampled_indices.extend(
            sampled_class
        )

    sampled_indices = np.array(
        sampled_indices
    )
    y_boot = y_test[
        sampled_indices
    ]
    pred_boot = y_pred_test[
        sampled_indices
    ]

    bootstrap_test_rows.append(
        {
            "QWK":
                cohen_kappa_score(
                    y_boot,
                    pred_boot,
                    weights="quadratic"
                ),
            "Macro F1":
                f1_score(
                    y_boot,
                    pred_boot,
                    average="macro",
                    zero_division=0
                ),
            "Accuracy":
                accuracy_score(
                    y_boot,
                    pred_boot
                ),
            "Macro Precision":
                precision_score(
                    y_boot,
                    pred_boot,
                    average="macro",
                    zero_division=0
                ),
            "Macro Recall":
                recall_score(
                    y_boot,
                    pred_boot,
                    average="macro",
                    zero_division=0
                )
        }
    )

bootstrap_test_df = pd.DataFrame(
    bootstrap_test_rows
)
print(
    "Repeticiones bootstrap:",
    len(bootstrap_test_df)
)

Repeticiones bootstrap: 5000


### 15.1. Construcción y persistencia de intervalos de confianza del 95 %

Se determinan los límites inferior y superior al 95 % de confianza (percentiles 2.5 % y 97.5 %) para cada métrica final de Test y se exportan a CSV.

In [25]:
point_estimates = {
    "QWK": test_qwk,
    "Macro F1": test_macro_f1,
    "Accuracy": test_accuracy,
    "Macro Precision": test_macro_precision,
    "Macro Recall": test_macro_recall
}

test_ci_rows = []
for metric, estimate in point_estimates.items():
    lower, upper = np.percentile(
        bootstrap_test_df[
            metric
        ],
        [2.5, 97.5]
    )
    test_ci_rows.append(
        {
            "Métrica":
                metric,
            "Estimación puntual":
                estimate,
            "IC 95% Inferior":
                lower,
            "IC 95% Superior":
                upper
        }
    )

test_confidence_intervals_df = (
    pd.DataFrame(
        test_ci_rows
    )
)
display(
    test_confidence_intervals_df.round(6)
)

bootstrap_test_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_bootstrap.csv",
    index=False
)
test_confidence_intervals_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "efficientnetb3_se_test_confidence_intervals.csv",
    index=False
)

,Métrica,Estimación puntual,IC 95% Inferior,IC 95% Superior
0,QWK,0.860161,0.824296,0.890705
1,Macro F1,0.626650,0.571383,0.678096
2,Accuracy,0.776718,0.746183,0.807252
3,Macro Precision,0.668083,0.615911,0.718757
4,Macro Recall,0.660466,0.603492,0.714714


## 16. Consolidación de resultados finales

La arquitectura EfficientNetB3+SE con Fine-Tuning fue seleccionada
exclusivamente mediante el conjunto Validation antes de utilizar Test.

La evaluación final se realizó una única vez sobre las 524 imágenes
reservadas de Test.

Los resultados de Test se utilizan exclusivamente para reportar el
rendimiento final del modelo y no generan modificaciones posteriores
sobre la arquitectura, sus pesos o el procedimiento de entrenamiento.

In [27]:
final_summary_df = pd.DataFrame(
    {
        "Métrica": [
            "QWK",
            "Macro F1",
            "Accuracy",
            "Macro Precision",
            "Macro Recall"
        ],
        "Validation": [
            0.877946,
            0.618500,
            0.780115,
            0.643499,
            0.648837
        ],
        "Test": [
            test_qwk,
            test_macro_f1,
            test_accuracy,
            test_macro_precision,
            test_macro_recall
        ]
    }
)
final_summary_df[
    "Diferencia Test - Validation"
] = (
    final_summary_df["Test"]
    - final_summary_df["Validation"]
)

display(
    final_summary_df.round(6)
)

final_summary_df.to_csv(
    COMPARISON_RESULTS_DIR
    / "rd_final_experiment_summary.csv",
    index=False
)

,Métrica,Validation,Test,Diferencia Test - Validation
0,QWK,0.877946,0.860161,-0.017785
1,Macro F1,0.618500,0.626650,0.008150
2,Accuracy,0.780115,0.776718,-0.003397
3,Macro Precision,0.643499,0.668083,0.024584
4,Macro Recall,0.648837,0.660466,0.011629


## 17. Actualización del registro de configuración definitivo

Se complementa el archivo de configuración del modelo seleccionado (`rd_selected_model_config.json`) con las métricas finales alcanzadas en Test, manteniendo intacta la justificación y los parámetros de selección tomados previamente.

In [28]:
import json

with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    final_config = json.load(f)

final_config[
    "final_test_evaluation"
] = {
    "images":
        int(len(test_df)),
    "QWK":
        float(test_qwk),
    "Macro F1":
        float(test_macro_f1),
    "Accuracy":
        float(test_accuracy),
    "Macro Precision":
        float(test_macro_precision),
    "Macro Recall":
        float(test_macro_recall),
    "bootstrap_repetitions":
        5000,
    "confidence_level":
        0.95,
    "used_for_model_selection":
        False
}

with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "Configuración final actualizada:"
)
print(
    FINAL_CONFIG_PATH
)

Configuración final actualizada:
/mnt/c/Users/SeuNg720/Documents/TP1-RetiScan/results/RD/model_comparison/rd_selected_model_config.json


## 18. Conclusiones del experimento

Se compararon ResNet50 y EfficientNetB3+SE para la clasificación ordinal
de cinco grados de Retinopatía Diabética utilizando las mismas
particiones, preprocesamiento, política de aumento de datos y condiciones
experimentales.

La selección de arquitectura se realizó exclusivamente sobre las 523
imágenes de Validation. ResNet50 obtuvo un QWK de 0.867617 y
EfficientNetB3+SE un QWK de 0.877946.

La diferencia puntual de QWK fue de 0.010330 a favor de
EfficientNetB3+SE. El análisis mediante bootstrap pareado estratificado
produjo un intervalo de confianza del 95 % de [-0.015200, 0.036515],
por lo que no se evidenció una diferencia clara entre ambas arquitecturas
dentro de la incertidumbre del conjunto Validation.

Siguiendo el criterio experimental definido previamente, que estableció
QWK como métrica principal, y considerando adicionalmente su menor
complejidad paramétrica, EfficientNetB3+SE con Fine-Tuning fue seleccionada
como arquitectura final.

Una vez congelada la selección, el modelo fue evaluado una única vez
sobre las 524 imágenes del conjunto Test.

En Test, EfficientNetB3+SE obtuvo:

- QWK: 0.860161, IC 95 % [0.824296, 0.890705].
- Macro F1: 0.626650, IC 95 % [0.571383, 0.678096].
- Accuracy: 0.776718, IC 95 % [0.746183, 0.807252].
- Macro Precision: 0.668083, IC 95 % [0.615911, 0.718757].
- Macro Recall: 0.660466, IC 95 % [0.603492, 0.714714].

Los resultados de Validation y Test muestran un comportamiento
consistente dentro de APTOS 2019. No obstante, se observan diferencias
relevantes en el rendimiento por clase, especialmente en los grados
Moderada, Severa y Proliferativa.

Estos resultados corresponden a una evaluación experimental interna
sobre APTOS 2019 y no constituyen por sí mismos una validación clínica
ni una evaluación de generalización sobre poblaciones externas.

El conjunto Test no será utilizado para realizar ajustes adicionales
sobre el modelo.